In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import numpy as np
import pandas as pd
import time

# ============================================================
# Corrected Anti-Diagonal Polynomial Experiment Code
# ============================================================
# This version replaces the incorrect Toeplitz-style construction
# A[i,j] = a[i+j].
#
# Correct convention:
#   f(x) = C(x)^T A C(x),   C(x) = [x^d, x^(d-1), ..., 1]^T
#
# Therefore, the coefficient of x^r is recovered by:
#   a_r = sum_{i+j = 2d-r} A[i,j]
#
# This anti-diagonal rule is used for:
#   1. polynomial reconstruction,
#   2. exact-collapse experiments,
#   3. rank-k surrogate reconstruction,
#   4. all result tables.
# ============================================================


# ============================================================
# 1. Basic polynomial utilities
# ============================================================

def poly_eval_desc(coeffs_desc, x):
    """Evaluate a polynomial with descending coefficients."""
    return np.polyval(coeffs_desc, x)


def poly_derivative_desc(coeffs_desc):
    """Derivative of a polynomial with descending coefficients."""
    n = len(coeffs_desc) - 1
    return np.array([coeffs_desc[i] * (n - i) for i in range(n)], dtype=float)


def newton_refine_roots(coeffs_desc, roots_init, max_iter=20, tol=1e-14):
    """Newton refinement of candidate roots on the original polynomial."""
    dcoeffs = poly_derivative_desc(coeffs_desc)
    refined = np.array(roots_init, dtype=np.complex128)

    for i, x0 in enumerate(refined):
        x = x0
        for _ in range(max_iter):
            fx = poly_eval_desc(coeffs_desc, x)
            dfx = poly_eval_desc(dcoeffs, x)

            if abs(dfx) < 1e-16:
                break

            x_new = x - fx / dfx
            if abs(x_new - x) < tol:
                x = x_new
                break

            x = x_new

        refined[i] = x

    return refined


def scale_invariant_backward_error(coeffs_desc, roots):
    """
    Scale-invariant backward error:
        eta(x) = |f(x)| / (||a||_1 * sum_j |x|^j)
    """
    coeffs_desc = np.asarray(coeffs_desc, dtype=np.complex128)
    n = len(coeffs_desc) - 1
    coeff_norm = max(np.sum(np.abs(coeffs_desc)), 1e-16)

    vals = []
    for x in roots:
        denom = coeff_norm * np.sum([abs(x) ** j for j in range(n + 1)])
        denom = max(denom, 1e-16)
        vals.append(abs(poly_eval_desc(coeffs_desc, x)) / denom)

    return float(np.max(vals))


# ============================================================
# 2. Correct anti-diagonal construction and reconstruction
# ============================================================

def anti_diagonal_poly_from_matrix(A, degree=None):
    """
    Reconstruct descending polynomial coefficients from A using:
        f(x) = C(x)^T A C(x), C(x) = [x^d, ..., 1]^T.

    If A is m x m, then d = m - 1 and degree <= 2d.

    Coefficient of x^r:
        a_r = sum_{i+j = 2d-r} A[i,j].
    """
    A = np.asarray(A, dtype=float)
    m = A.shape[0]
    d = m - 1
    max_degree = 2 * d

    coeffs_desc = np.zeros(max_degree + 1, dtype=float)

    for i in range(m):
        for j in range(m):
            power = 2 * d - (i + j)
            coeffs_desc[max_degree - power] += A[i, j]

    if degree is not None:
        # Keep only the requested polynomial degree.
        # This removes leading zero/cancelled terms for odd-degree exact-collapse cases.
        coeffs_desc = coeffs_desc[-(degree + 1):]

    return coeffs_desc


def canonical_anti_diagonal_matrix_from_poly(coeffs_desc):
    """
    Build a canonical symmetric anti-diagonal representative A_0 such that:
        f(x) = C(x)^T A_0 C(x)

    The coefficient a_r is distributed equally across all entries satisfying:
        i + j = 2d - r.

    This guarantees exact reconstruction under anti_diagonal_poly_from_matrix().
    """
    coeffs_desc = np.asarray(coeffs_desc, dtype=float)
    n = len(coeffs_desc) - 1
    d = int(np.ceil(n / 2))
    m = d + 1

    A = np.zeros((m, m), dtype=float)

    # coeffs_desc[0] corresponds to x^n.
    for idx, a_r in enumerate(coeffs_desc):
        power = n - idx
        anti_sum = 2 * d - power

        pairs = [(i, j) for i in range(m) for j in range(m) if i + j == anti_sum]
        if len(pairs) == 0:
            continue

        share = a_r / len(pairs)
        for i, j in pairs:
            A[i, j] += share

    return A


def coefficient_mismatch(coeffs_true_desc, coeffs_hat_desc):
    """Relative coefficient mismatch."""
    coeffs_true_desc = np.asarray(coeffs_true_desc, dtype=float)
    coeffs_hat_desc = np.asarray(coeffs_hat_desc, dtype=float)

    # Align lengths by left-padding the shorter vector.
    max_len = max(len(coeffs_true_desc), len(coeffs_hat_desc))
    a = np.pad(coeffs_true_desc, (max_len - len(coeffs_true_desc), 0))
    b = np.pad(coeffs_hat_desc, (max_len - len(coeffs_hat_desc), 0))

    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a), 1e-16))


# ============================================================
# 3. Rank diagnostics
# ============================================================

def singular_values_desc(A):
    return np.sort(np.linalg.svd(A, compute_uv=False))[::-1]


def numerical_rank(A, tol=1e-10):
    return int(np.linalg.matrix_rank(A, tol=tol))


def rho_sigma_k(A, k):
    s = singular_values_desc(A)
    if k >= len(s):
        return 0.0
    return float(s[k] / max(s[0], 1e-16))


def wedge_delta_tail(A, k):
    """
    Practical wedge-style rank certificate:
        product of singular values after k.
    Exact rank-k gives a near-zero value.
    """
    s = singular_values_desc(A)
    if k >= len(s):
        return 0.0
    return float(np.prod(s[k:]))


def spectral_rank_k_surrogate(A, k):
    """Symmetric spectral rank-k truncation."""
    evals, evecs = np.linalg.eigh(A)
    idx = np.argsort(np.abs(evals))[::-1]

    evals = evals[idx]
    evecs = evecs[:, idx]

    Uk = evecs[:, :k]
    Lk = np.diag(evals[:k])

    Ak = Uk @ Lk @ Uk.T
    return Ak, evals, evecs


# ============================================================
# 4. Exact-collapse polynomial generation
# ============================================================

def generate_exact_collapse(degree, rng):
    """
    Generate f(x) = Q1(x)^2 - Q2(x)^2 with:
        Q_i(x) = v_i^T C(x), C(x) = [x^d, ..., 1]^T.

    For odd degree n = 2d - 1, force leading x^(2d) cancellation
    by setting v2[0] = +/- v1[0].
    """
    n = degree
    d = int(np.ceil(n / 2))
    m = d + 1

    v1 = rng.normal(size=m)
    v2 = rng.normal(size=m)

    if n % 2 == 1:
        v2[0] = v1[0]

    A = np.outer(v1, v1) - np.outer(v2, v2)

    coeffs_full_desc = anti_diagonal_poly_from_matrix(A)
    coeffs_desc = coeffs_full_desc[-(n + 1):]

    return coeffs_desc, A, v1, v2


def roots_from_exact_collapse(v1, v2):
    """
    Roots of f = Q1^2 - Q2^2 are roots of:
        Q1 + Q2 = 0
        Q1 - Q2 = 0

    Since v vectors already match C(x)=[x^d,...,1],
    they are descending-order coefficients.
    """
    q_plus = v1 + v2
    q_minus = v1 - v2

    roots_plus = np.roots(q_plus)
    roots_minus = np.roots(q_minus)

    return np.concatenate([roots_plus, roots_minus])


# ============================================================
# 5. Table 1: Random synthetic rank-2 validation
# ============================================================

def run_random_rank2_validation(degrees=(5, 7, 20, 50), trials=20, seed=123):
    rng = np.random.default_rng(seed)
    rows = []

    for n in degrees:
        sig3_values = []
        delta_values = []
        eta_values = []
        rank_values = []
        m_values = []

        for _ in range(trials):
            coeffs_desc, A, v1, v2 = generate_exact_collapse(n, rng)
            roots = roots_from_exact_collapse(v1, v2)
            roots_ref = newton_refine_roots(coeffs_desc, roots)

            s = singular_values_desc(A)

            rank_values.append(numerical_rank(A))
            m_values.append(A.shape[0])
            sig3_values.append(s[2] if len(s) > 2 else 0.0)
            delta_values.append(wedge_delta_tail(A, 2))
            eta_values.append(scale_invariant_backward_error(coeffs_desc, roots_ref))

        rows.append({
            "Degree n": n,
            "Trials": trials,
            "m": int(np.median(m_values)),
            "rank(A)": int(np.median(rank_values)),
            "sigma_3(A)": np.median(sig3_values),
            "Delta_3(A)": np.median(delta_values),
            "eta_max_ref": np.median(eta_values),
        })

    return pd.DataFrame(rows)


# ============================================================
# 6. Table 2: Rank-k surrogate validation
# ============================================================

def run_rankk_surrogate_validation(degree=100, k_values=(2, 5, 10, 20), seed=456):
    rng = np.random.default_rng(seed)

    coeffs_desc, A0, v1, v2 = generate_exact_collapse(degree, rng)

    rows = []

    for k in k_values:
        start = time.perf_counter()

        Ak, evals, evecs = spectral_rank_k_surrogate(A0, k)
        coeffs_hat = anti_diagonal_poly_from_matrix(Ak, degree=degree)

        roots_hat = np.roots(coeffs_hat)
        roots_ref = newton_refine_roots(coeffs_desc, roots_hat)

        elapsed_ms = 1000 * (time.perf_counter() - start)

        rows.append({
            "k": k,
            "rho_sigma_k(A0)": rho_sigma_k(A0, k),
            "||f-fhat||/||f||": coefficient_mismatch(coeffs_desc, coeffs_hat),
            "eta_max": scale_invariant_backward_error(coeffs_desc, roots_hat),
            "eta_max_ref": scale_invariant_backward_error(coeffs_desc, roots_ref),
            "sketch_time_ms": elapsed_ms,
        })

    return pd.DataFrame(rows)


# ============================================================
# 7. Table 3: Large-scale validation
# ============================================================

def run_large_scale_validation(degrees=(100, 200, 400), seed=789):
    rng = np.random.default_rng(seed)
    rows = []

    for n in degrees:
        # Exact-collapse rank-2 case.
        start = time.perf_counter()
        coeffs_desc, A_exact, v1, v2 = generate_exact_collapse(n, rng)

        roots = roots_from_exact_collapse(v1, v2)
        roots_ref = newton_refine_roots(coeffs_desc, roots)

        coeffs_recon = anti_diagonal_poly_from_matrix(A_exact, degree=n)
        runtime = time.perf_counter() - start

        rows.append({
            "Degree n": n,
            "Regime": "exact-collapse",
            "Method": "rank-2 factor roots",
            "||f-fhat||/||f||": coefficient_mismatch(coeffs_desc, coeffs_recon),
            "rho_sigma_k": rho_sigma_k(A_exact, 2),
            "eta_max_ref": scale_invariant_backward_error(coeffs_desc, roots_ref),
            "Runtime_s": runtime,
        })

        # Generic anti-diagonal matrix case.
        start = time.perf_counter()
        coeffs_generic = rng.normal(size=n + 1)
        coeffs_generic = coeffs_generic / max(np.linalg.norm(coeffs_generic), 1e-16)

        A_generic = canonical_anti_diagonal_matrix_from_poly(coeffs_generic)

        k = min(20, A_generic.shape[0])
        Ak, _, _ = spectral_rank_k_surrogate(A_generic, k)
        coeffs_hat = anti_diagonal_poly_from_matrix(Ak, degree=n)

        roots_hat = np.roots(coeffs_hat)
        roots_ref = newton_refine_roots(coeffs_generic, roots_hat)

        runtime = time.perf_counter() - start

        rows.append({
            "Degree n": n,
            "Regime": "generic",
            "Method": f"spectral rank-{k} surrogate",
            "||f-fhat||/||f||": coefficient_mismatch(coeffs_generic, coeffs_hat),
            "rho_sigma_k": rho_sigma_k(A_generic, k),
            "eta_max_ref": scale_invariant_backward_error(coeffs_generic, roots_ref),
            "Runtime_s": runtime,
        })

    return pd.DataFrame(rows)


# ============================================================
# 8. Table 4: Benchmark polynomial experiments
# ============================================================

def wilkinson_coeffs(n=20):
    """Wilkinson polynomial with roots 1, 2, ..., n."""
    return np.poly(np.arange(1, n + 1))


def chebyshev_coeffs(n=40):
    """Chebyshev polynomial T_n coefficients in descending powers."""
    from numpy.polynomial import Chebyshev, Polynomial
    T = Chebyshev.basis(n)
    P = T.convert(kind=Polynomial)
    coeffs_asc = P.coef
    return coeffs_asc[::-1]


def run_benchmark_validation():
    rows = []

    benchmarks = [
        ("Wilkinson", 20, wilkinson_coeffs(20)),
        ("Chebyshev", 40, chebyshev_coeffs(40)),
    ]

    for name, n, coeffs_desc in benchmarks:
        coeffs_desc = np.asarray(coeffs_desc, dtype=float)
        coeffs_desc = coeffs_desc / max(np.linalg.norm(coeffs_desc), 1e-16)

        A0 = canonical_anti_diagonal_matrix_from_poly(coeffs_desc)

        # Companion baseline
        start = time.perf_counter()
        roots_comp = np.roots(coeffs_desc)
        runtime_comp = time.perf_counter() - start

        rows.append({
            "Benchmark": name,
            "Degree n": n,
            "rho_sigma_2(A0)": rho_sigma_k(A0, 2),
            "Method": "companion",
            "eta_max": scale_invariant_backward_error(coeffs_desc, roots_comp),
            "Runtime_s": runtime_comp,
        })

        # Rank-aware surrogate baseline
        start = time.perf_counter()
        k = min(20, A0.shape[0])
        Ak, _, _ = spectral_rank_k_surrogate(A0, k)
        coeffs_hat = anti_diagonal_poly_from_matrix(Ak, degree=n)
        roots_hat = np.roots(coeffs_hat)
        roots_ref = newton_refine_roots(coeffs_desc, roots_hat)
        runtime_rank = time.perf_counter() - start

        rows.append({
            "Benchmark": name,
            "Degree n": n,
            "rho_sigma_2(A0)": rho_sigma_k(A0, 2),
            "Method": f"rank-aware surrogate k={k}",
            "eta_max": scale_invariant_backward_error(coeffs_desc, roots_ref),
            "Runtime_s": runtime_rank,
        })

    return pd.DataFrame(rows)


# ============================================================
# 9. Table 5: Matrix-pencil eigenvalue validation
# ============================================================

def eigenpair_residual_linear(M, eigvals, eigvecs):
    vals = []
    Mnorm = np.linalg.norm(M, 2)

    for i, lam in enumerate(eigvals):
        z = eigvecs[:, i]
        num = np.linalg.norm((M - lam * np.eye(M.shape[0])) @ z)
        den = (Mnorm + abs(lam)) * max(np.linalg.norm(z), 1e-16)
        vals.append(num / max(den, 1e-16))

    return float(np.max(vals))


def run_matrix_pencil_validation(seed=321):
    rng = np.random.default_rng(seed)
    rows = []

    # Random symmetric linear pencil
    for experiment, m, symmetric in [
        ("Random symmetric", 100, True),
        ("Random nonnormal", 100, False),
    ]:
        start = time.perf_counter()

        M = rng.normal(size=(m, m))
        if symmetric:
            M = (M + M.T) / 2

        eigvals, eigvecs = np.linalg.eig(M)
        residual = eigenpair_residual_linear(M, eigvals, eigvecs)
        runtime = time.perf_counter() - start

        rows.append({
            "Experiment": experiment,
            "Dimension m": m,
            "Pencil type": "linear",
            "max_r_eig": residual,
            "Runtime_s": runtime,
        })

    # Quadratic pencil placeholder-style numerical test via companion linearization
    m = 30
    start = time.perf_counter()

    K = rng.normal(size=(m, m))
    C = rng.normal(size=(m, m))
    M = np.eye(m)

    Z = np.zeros((m, m))
    I = np.eye(m)

    # Linearization:
    # [0 I; -K -C] y = lambda [I 0; 0 M] y
    # Since M=I, solve standard eigenproblem.
    L = np.block([[Z, I], [-K, -C]])
    eigvals, eigvecs = np.linalg.eig(L)

    residuals = []
    for idx, lam in enumerate(eigvals):
        z = eigvecs[:m, idx]
        Pz = (K + lam * C + (lam ** 2) * M) @ z
        den = (
            np.linalg.norm(K, 2)
            + abs(lam) * np.linalg.norm(C, 2)
            + abs(lam) ** 2 * np.linalg.norm(M, 2)
        ) * max(np.linalg.norm(z), 1e-16)
        residuals.append(np.linalg.norm(Pz) / max(den, 1e-16))

    runtime = time.perf_counter() - start

    rows.append({
        "Experiment": "Quadratic pencil",
        "Dimension m": m,
        "Pencil type": "quadratic",
        "max_r_eig": float(np.max(residuals)),
        "Runtime_s": runtime,
    })

    return pd.DataFrame(rows)


# ============================================================
# 10. Runtime scaling table
# ============================================================

def run_runtime_scaling(degrees=(100, 200, 400), k=20, seed=654):
    rng = np.random.default_rng(seed)
    rows = []

    for n in degrees:
        coeffs = rng.normal(size=n + 1)
        coeffs = coeffs / max(np.linalg.norm(coeffs), 1e-16)

        A0 = canonical_anti_diagonal_matrix_from_poly(coeffs)
        m = A0.shape[0]

        start = time.perf_counter()
        _ = np.linalg.eigh(A0)
        full_time_ms = 1000 * (time.perf_counter() - start)

        start = time.perf_counter()
        _ = spectral_rank_k_surrogate(A0, min(k, m))
        sketch_time_ms = 1000 * (time.perf_counter() - start)

        rows.append({
            "Degree n": n,
            "m": m,
            "k": min(k, m),
            "full_eig_time_ms": full_time_ms,
            "sketch_QR_time_ms": sketch_time_ms,
        })

    return pd.DataFrame(rows)


# ============================================================
# 11. Export all manuscript tables
# ============================================================

def format_scientific_df(df):
    """Return a copy formatted as manuscript-readable strings."""
    out = df.copy()

    for col in out.columns:
        if pd.api.types.is_float_dtype(out[col]):
            out[col] = out[col].map(lambda x: f"{x:.3e}")

    return out


def export_all_tables(output_prefix="anti_diagonal_experiment_results"):
    tables = {
        "random_rank2_validation": run_random_rank2_validation(),
        "large_scale_validation": run_large_scale_validation(),
        "rankk_validation": run_rankk_surrogate_validation(),
        "benchmark_polynomials": run_benchmark_validation(),
        "matrix_pencil_experiments": run_matrix_pencil_validation(),
        "runtime_scaling": run_runtime_scaling(),
    }

    # Export raw numeric tables to Excel.
    xlsx_path = f"{output_prefix}.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        for name, df in tables.items():
            df.to_excel(writer, sheet_name=name[:31], index=False)

    # Export CSV files and LaTeX files.
    for name, df in tables.items():
        df.to_csv(f"{output_prefix}_{name}.csv", index=False)

        formatted = format_scientific_df(df)
        latex = formatted.to_latex(index=False, escape=False)
        with open(f"{output_prefix}_{name}.tex", "w", encoding="utf-8") as f:
            f.write(latex)

    return tables


# ============================================================
# 12. Main execution
# ============================================================

if __name__ == "__main__":
    tables = export_all_tables()

    print("\nGenerated manuscript tables using corrected anti-diagonal construction.\n")

    for name, df in tables.items():
        print("=" * 80)
        print(name)
        print("=" * 80)
        print(format_scientific_df(df).to_string(index=False))
        print()



Generated manuscript tables using corrected anti-diagonal construction.

random_rank2_validation
 Degree n  Trials  m  rank(A) sigma_3(A) Delta_3(A) eta_max_ref
        5      20  4        2  1.844e-16  8.071e-33   7.981e-18
        7      20  5        2  3.408e-16  7.506e-49   1.079e-17
       20      20 11        2  1.254e-15 1.682e-142   6.170e-18
       50      20 26        2  3.176e-15  0.000e+00   3.573e-18

large_scale_validation
 Degree n         Regime                     Method ||f-fhat||/||f|| rho_sigma_k eta_max_ref Runtime_s
      100 exact-collapse        rank-2 factor roots        0.000e+00   1.290e-16   3.320e-18 2.472e-02
      100        generic spectral rank-20 surrogate        3.571e-01   3.526e-01   1.108e-03 2.137e-01
      200 exact-collapse        rank-2 factor roots        0.000e+00   9.849e-17   2.442e-18 1.270e-01
      200        generic spectral rank-20 surrogate        5.582e-01   2.126e-01   8.210e-04 7.873e-01
      400 exact-collapse        rank-2 fact